# 01 - Ingest Public Datasets

Objectif: charger des datasets publics (sans entrainement), normaliser le schema et sauvegarder dans `data/classifier_dataset/raw_sources/`.

In [1]:
from pathlib import Path
import pandas as pd

BASE = Path('../data/classifier_dataset/raw_sources')
for d in ['medquad', 'mediqal', 'frenchmedmcqa', 'parhaf', 'manual_icu', 'red_team']:
    (BASE / d).mkdir(parents=True, exist_ok=True)

COMMON_COLS = ['text_raw', 'source_dataset', 'source_type', 'is_synthetic', 'language']

## Load MedQuAD

In [2]:
from datasets import load_dataset

medquad = load_dataset('lavita/MedQuAD', split='train')
medquad_df = pd.DataFrame({
    'text_raw': [str(x.get('question', '')).strip() for x in medquad],
    'source_dataset': 'medquad',
    'source_type': 'public_medical_qa',
    'is_synthetic': False,
    'language': 'en',
})
medquad_df = medquad_df[medquad_df['text_raw'] != ''].drop_duplicates()
medquad_df.to_csv(BASE / 'medquad' / 'ingested.csv', index=False)
print('MedQuAD rows:', len(medquad_df))

c:\Users\223114186\Downloads\Multimodal_Speech_&_Emotion_Analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MedQuAD rows: 44603


## Load Other Sources (placeholders if unavailable)

In [5]:
def safe_dataset_load_any(candidates, language='fr', source_type='public_medical_exam'):
    for name in candidates:
        try:
            ds = load_dataset(name, split='train')
            text_col = None
            for c in ['question', 'Question', 'input', 'text']:
                if c in ds.column_names:
                    text_col = c
                    break
            if text_col is None:
                continue
            out = pd.DataFrame({
                'text_raw': [str(x.get(text_col, '')).strip() for x in ds],
                'source_dataset': name,
                'source_type': source_type,
                'is_synthetic': False,
                'language': language,
            })
            out = out[out['text_raw'] != ''].drop_duplicates()
            return out
        except Exception:
            continue
    return pd.DataFrame(columns=COMMON_COLS)

mediqal_df = safe_dataset_load_any(
    ['MediQAl/mediqal', 'mediqal/mediqal', 'MediQAl'],
    language='fr',
    source_type='french_medical_exam',
)
frenchmedmcqa_df = safe_dataset_load_any(
    ['BSC-LT/FrenchMedMCQA'],
    language='fr',
    source_type='french_medical_exam',
)
parhaf_df = pd.DataFrame(columns=COMMON_COLS)

mediqal_df.to_csv(BASE / 'mediqal' / 'ingested.csv', index=False)
frenchmedmcqa_df.to_csv(BASE / 'frenchmedmcqa' / 'ingested.csv', index=False)
parhaf_df.to_csv(BASE / 'parhaf' / 'ingested.csv', index=False)

print('MediQAl rows:', len(mediqal_df))
print('FrenchMedMCQA rows:', len(frenchmedmcqa_df))
print('PARHAF rows:', len(parhaf_df), '(placeholder until licensed/local source is added)')

MediQAl rows: 0
FrenchMedMCQA rows: 0
PARHAF rows: 0 (placeholder until licensed/local source is added)


## Source index

In [6]:
index_df = pd.DataFrame([
    {'source_dataset': 'medquad', 'path': str(BASE / 'medquad' / 'ingested.csv')},
    {'source_dataset': 'mediqal', 'path': str(BASE / 'mediqal' / 'ingested.csv')},
    {'source_dataset': 'frenchmedmcqa', 'path': str(BASE / 'frenchmedmcqa' / 'ingested.csv')},
    {'source_dataset': 'parhaf', 'path': str(BASE / 'parhaf' / 'ingested.csv')},
])
index_df.to_csv(BASE / 'source_index.csv', index=False)
index_df

,source_dataset,path
0,medquad,..\data\classifier_dataset\raw_sources\medquad...
1,mediqal,..\data\classifier_dataset\raw_sources\mediqal...
2,frenchmedmcqa,..\data\classifier_dataset\raw_sources\frenchm...
3,parhaf,..\data\classifier_dataset\raw_sources\parhaf\...
